# 02 — Prepare ligands

End-to-end ligand prep for an input SMILES library:

1. **Standardise** — neutralise charges, desalt, canonicalise tautomers.
2. **Compute properties** — MW, LogP, HBD/HBA, TPSA, rotatable bonds, QED, rings.
3. **Drug-likeness gate** — Lipinski (Ro5), Veber, QED ≥ 0.5.
4. **PAINS flag** — pan-assay interference compound check.
5. **3D embed** — ETKDGv3 conformer + MMFF (UFF fallback) optimisation.
6. **Write SDF** — ready for docking.

**Runtime (serial):** ~1 min for 500 compounds, ~10 min for 3.7k. Use `n_workers > 1` outside Jupyter on Windows.

**Done signal:** Lipinski pass rate ≈ 70–90% on a real-world library.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !pip install -q rdkit datamol "prolif>=2.0" posebusters meeko py3Dmol biopython scikit-learn xgboost lightgbm
    REPO_ROOT = Path("/content/aidd-pipeline")
    if not REPO_ROOT.exists():
        !git clone https://github.com/hvmarco/aidd-pipeline.git {REPO_ROOT}
    sys.path.insert(0, str(REPO_ROOT / "src"))
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Repo root: {REPO_ROOT}")
print(f"Running on: {'Colab' if IS_COLAB else 'local'}")

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import py3Dmol
from rdkit import Chem

from aidd.ligands import (
    read_smiles, prepare, prepare_library, write_sdf,
    LIPINSKI_RULES, VEBER_RULES,
)

sns.set_theme(style="whitegrid")
print("setup ok")

## 2. Inputs

Default is the **small** ERK2 library (3.7k compounds). Switch to `training.smi` for the full 47k. The notebook then processes a `SAMPLE_N` subset by default.

In [ ]:
INPUT_SMI = REPO_ROOT / "data" / "compounds" / "erk2" / "training_small.smi"
LABELS    = REPO_ROOT / "data" / "labels" / "erk2_training.tsv"
OUTPUT_SDF = REPO_ROOT / "data" / "derived" / "erk2" / "ligands_prepared.sdf"

SAMPLE_N = 500   # number of compounds to process in this run (set to None for all)
N_WORKERS = 1    # bump to 4-8 outside Jupyter; spawn issues in Jupyter on Windows mean serial is safest

assert INPUT_SMI.exists(), f"missing {INPUT_SMI}"
print(f"Input:  {INPUT_SMI.relative_to(REPO_ROOT)}")
print(f"Output: {OUTPUT_SDF.relative_to(REPO_ROOT)}")

In [ ]:
df_in = read_smiles(INPUT_SMI)
print(f"Loaded {len(df_in):,} compounds")
df_in.head()

## 3. Run the prep pipeline

Each compound is standardised, scored on properties, PAINS-checked, and 3D-embedded. Progress bar shows ETA.

In [ ]:
df_subset = df_in.sample(n=min(SAMPLE_N, len(df_in)), random_state=42).reset_index(drop=True) if SAMPLE_N else df_in
print(f"Preparing {len(df_subset):,} compounds…")

df = prepare_library(df_subset, n_workers=N_WORKERS, progress=True)
print(f"Done. ok={df['ok'].sum():,}/{len(df):,}  ({df['ok'].mean():.0%} success)")

## 4. Property distributions

Sanity-check the library is in the drug-discovery sweet spot. Outliers here = candidates the gate will trim.

In [ ]:
ok = df[df["ok"]].copy()

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
props = [
    ("mw",       "Molecular weight (Da)", 500),
    ("logp",     "LogP",                  5),
    ("hbd",      "H-bond donors",         5),
    ("hba",      "H-bond acceptors",     10),
    ("tpsa",     "Polar surface area (Å²)", 140),
    ("qed",      "QED (drug-likeness)",  0.5),
]
for ax, (col, label, threshold) in zip(axes.flat, props):
    sns.histplot(ok[col], ax=ax, bins=30, color="steelblue")
    ax.axvline(threshold, color="crimson", linestyle="--", label=f"Ro5/QED gate ≈ {threshold}")
    ax.set_xlabel(label)
    ax.legend(fontsize=8)
fig.suptitle("Property distributions (prepared library)")
fig.tight_layout()
plt.show()

## 5. Gate pass rates

In [ ]:
rates = {
    "Standardise + 3D embed": df["ok"].mean(),
    "Lipinski (Ro5) pass":    ok["lipinski_pass"].mean(),
    "Veber pass":             ok["veber_pass"].mean(),
    "QED ≥ 0.5":              ok["qed_pass"].mean(),
    "PAINS-clean":            (ok["pains"] == "").mean(),
}
pass_all = ok["lipinski_pass"] & ok["veber_pass"] & ok["qed_pass"] & (ok["pains"] == "")
rates["All gates combined"] = pass_all.mean()

pd.Series(rates, name="pass_rate").map("{:.1%}".format).to_frame()

In [ ]:
# Top PAINS rule matches (when present)
pains_hits = ok.loc[ok["pains"] != "", "pains"].value_counts().head(10)
if len(pains_hits):
    print("Most common PAINS matches:")
    print(pains_hits.to_string())
else:
    print("No PAINS hits in this sample.")

## 6. Visualise one prepared 3D structure

Confirm the embed produced something sensible.

In [ ]:
# pick the highest-QED PAINS-free compound
pick = ok[(ok["pains"] == "") & ok["lipinski_pass"]].sort_values("qed", ascending=False).head(1)
mol = pick.iloc[0]["mol"]
print(f"Showing {pick.iloc[0]['name'] or 'compound'}  —  QED {pick.iloc[0]['qed']:.2f}, MW {pick.iloc[0]['mw']:.0f}")

viewer = py3Dmol.view(width=500, height=400)
viewer.addModel(Chem.MolToMolBlock(mol), "mol")
viewer.setStyle({"stick": {"colorscheme": "cyanCarbon"}})
viewer.zoomTo()
viewer.show()

## 7. Cross-reference with labels (optional)

Sanity check that the drug-likeness gate isn't preferentially trimming known **actives**. If actives pass at a much lower rate than inactives, the gate is too aggressive for this target. Only runs if compound IDs match the labels file.

In [ ]:
if LABELS.exists():
    labels = pd.read_csv(LABELS, sep=r"\s+")
    labels["CPD_ID"] = labels["CPD_ID"].astype(str)
    n_match = ok["name"].isin(labels["CPD_ID"]).sum()
    if n_match == 0:
        print("No name overlap between this SMILES file and the labels — skip the cross-reference.")
        print("(This is expected for training_small.smi, whose names use the 'ERKxxx' format,")
        print(" whereas the labels file uses numeric CPD_IDs from training.smi.)")
    else:
        merged = ok.merge(labels, left_on="name", right_on="CPD_ID", how="inner")
        gate_pass = merged["lipinski_pass"] & merged["veber_pass"] & merged["qed_pass"] & (merged["pains"] == "")
        merged["gate_pass"] = gate_pass
        table = (
            merged.groupby("Active")["gate_pass"]
            .agg(["size", "sum", "mean"])
            .rename(columns={"size": "count", "sum": "passed", "mean": "pass_rate"})
        )
        table["pass_rate"] = table["pass_rate"].map("{:.1%}".format)
        print(f"Matched {n_match:,} compounds to labels.")
        display(table)
else:
    print(f"Labels file not found at {LABELS}; skipping cross-reference.")

## 8. Write SDF for downstream docking

In [ ]:
passing = ok[
    ok["lipinski_pass"] & ok["veber_pass"] & ok["qed_pass"] & (ok["pains"] == "")
].copy()

n = write_sdf(
    passing,
    OUTPUT_SDF,
    mol_col="mol",
    id_col="name",
    props_to_write=[
        "smiles_std", "mw", "logp", "hbd", "hba", "tpsa", "rotbonds", "qed",
        "lipinski_violations", "veber_violations",
    ],
)
print(f"Wrote {n:,} prepared molecules → {OUTPUT_SDF.relative_to(REPO_ROOT)}")

## 9. Recap & next steps

What this notebook produced:
- `data/derived/erk2/ligands_prepared.sdf` — compounds that pass every gate, with 3D coordinates ready for docking.
- Property + drug-likeness distributions and a PAINS audit.
- Optional cross-check against activity labels.

**Next:** `notebooks/03_dock_gnina.ipynb` — gnina docking + PoseBusters QC against the ERK2 receptor (step 9 in the plan).

**To prep the full library** (47k for `training.smi`) outside Jupyter, run as a script with parallel workers:

```python
from pathlib import Path
from aidd.ligands import read_smiles, prepare_library, write_sdf
df = read_smiles("data/compounds/erk2/training.smi")
out = prepare_library(df, n_workers=8, progress=True)
write_sdf(out[out['ok']], "data/derived/erk2/ligands_prepared_full.sdf",
          props_to_write=['smiles_std','mw','logp','qed','lipinski_pass'])
```